## Cell 1 — Define `get_weather()` Tool Function

Imports the `requests` library and defines a helper function `get_weather(city)` that:
- Calls the free [wttr.in](https://wttr.in) weather API with the city name
- Parses the JSON response
- Extracts the current temperature in °C from `current_condition[0]["temp_C"]`
- Returns a human-readable string like `"The temperature in Delhi is 31°C"`

This function will later be **called by the agent** when the LLM decides to invoke the weather tool.

In [4]:
import requests

def get_weather(city):

    url = "https://wttr.in/" + city + "?format=j1"

    response = requests.get(url)

    data = response.json()

    temperature = data["current_condition"][0]["temp_C"]

    return f"The temperature in {city} is {temperature}°C"

c:\Users\yashi\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


## Cell 2 — Sanity-Test the Weather Function

A quick smoke-test that calls `get_weather("Delhi")` directly (no LLM involved yet) and prints the result.  
Expected output: `The temperature in Delhi is 31°C`  
This confirms the API call and JSON parsing work correctly before wiring the function into an agent.

In [5]:
result = get_weather("Delhi")

print(result)

The temperature in Delhi is 31°C


# This is the real agent

## Cell 3 — Define the Tool Schema (`weather_tool`)

Creates a **tool descriptor dictionary** in the OpenAI function-calling format.  
This JSON schema tells the LLM:
- **Name**: `get_weather` — the function to invoke
- **Description**: a plain-English hint so the model knows *when* to use it
- **Parameters**: expects a single required argument `city` (a string)

This object is passed to the API in the `tools=` parameter so the model can decide to call the function.

In [6]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current temperature of a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Name of the city"
                }
            },
            "required": ["city"]
        }
    }
}

## Cell 4 — Load Environment Variables

Loads credentials from a `.env` file using `python-dotenv` and confirms the Groq API key is present.  
- `load_dotenv()` reads `.env` into `os.environ`
- `bool(os.getenv("GROQ_API_KEY"))` prints `True` if the key was found, keeping the actual secret hidden from output

The `GROQ_API_KEY` is required to authenticate requests to the Groq LLM backend.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API key loaded:", bool(os.getenv("GROQ_API_KEY")))

API key loaded: True


## Cell 5 — First LLM Call (Tool Selection)

Initialises an `OpenAI` client pointed at **Groq's OpenAI-compatible endpoint** (`https://api.groq.com/openai/v1`) and sends the user query `"What is the weather in Delhi?"` along with the `weather_tool` schema.

Key points:
- Groq's API is compatible with the OpenAI Python SDK — only `base_url` and `api_key` differ
- The model responds with a **tool call** (not a text answer) because it recognises the query matches the weather tool
- `response.choices[0].message` holds the model reply, which contains `tool_calls` instead of plain `content`

In [8]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = os.getenv("MODEL", "openai/gpt-oss-120b")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What is the weather in Delhi?"
        }
    ],
    tools=[weather_tool]
)

message = response.choices[0].message

## Cell 6 — Inspect the Model's Tool Call Decision

Checks whether the model returned a **tool call** or a plain text reply:
- If `message.tool_calls` is non-empty → prints each tool name and its JSON arguments (e.g. `{"city": "Delhi"}`)
- Otherwise → prints the model's direct text content

For a weather question with the tool available, the model should always choose the tool path, confirming the function-calling mechanism is working.

In [9]:
if message.tool_calls:
    for call in message.tool_calls:
        print("Tool:", call.function.name)
        print("Arguments:", call.function.arguments)
else:
    print("Model:", message.content)

Tool: get_weather
Arguments: {"city":"Delhi"}


## Cell 7 — Execute the Tool and Get the Real Result

Acts as the **tool executor / dispatcher**:
1. Extracts the first tool call from the model's message
2. Reads the `tool_name` and deserialises the JSON arguments with `json.loads()`
3. Routes to the correct Python function (`get_weather`) using a simple `if` check
4. Calls `get_weather(city)` with the extracted city name and stores the live result

This is the **bridge** between the model's *intent* (tool call) and the *real-world action* (HTTP request to wttr.in).

In [10]:
import json

if message.tool_calls:

    call = message.tool_calls[0]

    tool_name = call.function.name
    arguments = json.loads(call.function.arguments)

    if tool_name == "get_weather":
        result = get_weather(arguments["city"])

    print("Tool result:", result)

Tool result: The temperature in Delhi is 31°C


## Cell 8 — Build the Full Conversation History

Reconstructs the message thread that will be sent back to the LLM so it can formulate a **final natural-language answer**:

| Role | Content |
|------|---------|
| `user` | Original question: *"What is the weather in Delhi?"* |
| `assistant` | The model's tool-call message (`message`) |
| `tool` | The actual result returned by `get_weather`, linked to the call via `tool_call_id` |

Including the `tool_call_id` is **required** by the API to correctly associate the tool result with the specific function invocation.

In [11]:
messages = [
    {
        "role": "user",
        "content": "What is the weather in Delhi?"
    },
    message
]

messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": result
})

## Cell 9 — Second LLM Call (Final Answer Generation)

Sends the **full conversation history** (user → assistant tool-call → tool result) back to the LLM for a second inference pass.  
The model now has real weather data in context and generates a concise, human-friendly reply such as:  
`The current temperature in Delhi is **31 °C**.`

This two-step pattern — **(1) tool selection → (2) answer generation** — is the core of the **ReAct / function-calling agent loop**.

In [12]:
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[weather_tool]
)

print(final_response.choices[0].message.content)

The current temperature in Delhi is **31 °C**.
